# Write a Metropolis-Hastings MCMC algorithm

In this notebook you will learn the logic of a Metropolis-Hastings Markov chain Monte Carlo algorithm and write your own.

Placeholder functions are given below, but they are empty. Function definitions tell you what the content of these functions should be, but you have to write them.

The overarching function that brings these modular functions together is **provided**, but I strongly encourage you to read and understand this function as well. Once you write the modular functions, run the main function on the provided toy model and get constraints.

In fact, spend time parsing every cell, even the provided ones. 

We will then spend some time building intuition for MH MCMCs to help you make more informed choices when running chains and understanding their output. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
import corner
from tqdm.auto import tqdm

np.random.seed(42)

mpl.rcParams['figure.dpi']      = 110
mpl.rcParams['axes.labelsize']  = 12
mpl.rcParams['font.size']       = 11
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['axes.spines.top']   = False
mpl.rcParams['axes.spines.right'] = False

## Orientation

From the lecture, Bayes' theorem gives us the posterior (the probability of our parameters given the observed data):

$$P(\theta \mid D) \propto \mathcal{L}(D \mid \theta) \times \pi(\theta)$$

The likelihood $\mathcal{L}$ (probability of the data given parameters) and prior $\pi$ (probability of the parameters) are **black boxes** handed to the sampler. The sampler does not need to know what problem it is solving.

Its only job: **draw samples from** $P(\theta \mid D)$ without evaluating it everywhere, therefore improving over simply grid-sampling which becomes prohibitive in large-dimensional parameter spaces. 

This session builds the machinery for the sampler.

---
## Section 1 — A toy posterior: the 2D correlated Gaussian

Before touching real data we need a case where the answer is **known analytically**, so we can verify our sampler is correct.

Our toy posterior is a 2D Gaussian with:
- Mean: $\boldsymbol{\mu} = [3,\,-1]$
- Marginal widths: $\sigma_1 = 1$, $\sigma_2 = 2$
- Correlation: $\rho = 0.8$

The elongated, tilted shape makes pathological step-size behaviour easy to diagnose.

In [ ]:
# --- PROVIDED: toy log-posterior ---
# This is the black box the sampler will call.
# You do NOT need to understand its internals today.

MU_TOY  = np.array([3.0, -1.0])
COV_TOY = np.array([[1.0, 1.6],
                    [1.6, 4.0]])   # sigma1=1, sigma2=2, rho=0.8

_COV_INV  = np.linalg.inv(COV_TOY)
_LOG_NORM = -0.5 * np.log(np.linalg.det(2 * np.pi * COV_TOY))

def log_posterior_toy(theta):
    """
    Log-posterior of a 2D correlated Gaussian.

    Parameters
    ----------
    theta : array-like, shape (2,)

    Returns
    -------
    float
    """
    d = np.asarray(theta, dtype=float) - MU_TOY
    return float(_LOG_NORM - 0.5 * d @ _COV_INV @ d)

In [ ]:
# --- PROVIDED: visualise the toy posterior ---

# How finely should we grid? 
N_grid = 200
# How does N_grid affect the accuracy of the peak location and the contour levels? 
# Try changing it to 30 or 1000 and see what happens.

# grid of theta values 
t1 = np.linspace(-1.5, 7.5, N_grid)
t2 = np.linspace(-8.0, 6.0, N_grid)
T1, T2 = np.meshgrid(t1, t2)

# calculate log-posterior on the grid
logp_grid = np.vectorize(lambda a, b: log_posterior_toy([a, b]))(T1, T2)
peak   = logp_grid.max()
levels = peak - np.array([4.5, 2.0, 0.5])   # ~3σ, ~2σ, ~1σ (must be increasing)

# get index of max log postterior
idx      = np.unravel_index(np.argmax(logp_grid), logp_grid.shape)
peak_t1  = T1[idx]
peak_t2  = T2[idx]

fig, ax = plt.subplots(figsize=(6, 5))
# plot filled contours and contour lines of the log-posterior
ax.contourf(T1, T2, logp_grid, levels=levels, alpha=0.35, cmap='Blues')
ax.contour( T1, T2, logp_grid, levels=levels, colors='steelblue', linewidths=1.5)
# plot the true mean and the grid peak
ax.scatter(*MU_TOY, color='darkorchid', s=80, zorder=5,
           label=f'True mean {list(MU_TOY)}')
ax.scatter(peak_t1, peak_t2, color='darkorange', marker='*', s=120, zorder=6,
           facecolors='gold', label=f'Grid peak ({peak_t1:.2f}, {peak_t2:.2f})')

ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel(r'$\theta_2$')
ax.set_title('Toy 2D Gaussian posterior')
ax.legend(loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

---
## Section 2 — The Metropolis-Hastings algorithm

The MH algorithm builds a Markov chain whose stationary distribution *is* the posterior. Here is the full algorithm:

```
1. Start at theta_current          (your initial guess)
2. Propose   theta*  ~  Normal(theta_current, step_size)
3. Compute   log_alpha = log_post(theta*) - log_post(theta_current)
4. Draw      u  ~  Uniform(0, 1)
5. If  log(u) < log_alpha:   ACCEPT  -->  theta_current = theta*
   Else:                     REJECT  -->  theta_current stays
                                          (record it again!)
6. Store theta_current.  Go to step 2.
```

**Key intuitions:**
- Step 5 always accepts when the proposed point has *higher* posterior (log\_alpha > 0).
- Sometimes you accept a *worse* point — this lets the chain explore and escape local minima.
- Use `log(u) < log_alpha`, **not** `u < alpha`. Subtracting logs avoids floating-point underflow.
- Rejection means you **store the current point again**, not skip a step. Every iteration produces one sample.
- The **proposal width** (`step_size`) controls how far you jump. Too narrow: the chain barely moves. Too wide: almost every proposal is rejected.

> **Common mistake:** thinking ‘accept if better, reject if worse’. This would produce a hill-climber, not a sampler. Occasional downhill acceptance is required to **explore** the region.

We implement this as **three modular functions** (which you write), then wire them into a single master `run_mh_chain` function (provided).

---
## Section 3 — Build the modular functions

Fill in the body of each function. The docstring tells you exactly what to return.

> **Tip:** each function body is 1–5 lines. If yours is longer, you are probably over-complicating it.

In [ ]:
# --- STUDENT (3a): proposal distribution ---

def propose(theta_current, step_size):
    """
    Draw a candidate next position from a Gaussian proposal
    centred on the current position.

    Parameters
    ----------
    theta_current : np.ndarray, shape (n_params,)
        Current position in parameter space.
    step_size : float or np.ndarray
        Standard deviation of the Gaussian proposal.
        Scalar => isotropic. Array => independent per-parameter widths.

    Returns
    -------
    theta_proposed : np.ndarray, shape (n_params,)
    """
    pass   # ~1-2 lines using np.random.normal

In [ ]:
# --- STUDENT (3b): acceptance probability ---

def acceptance_probability(log_post_proposed, log_post_current):
    """
    Compute the log MH acceptance probability.

    The MH acceptance probability is
        alpha = min(1, exp(log_post_proposed - log_post_current))
    Return the *log* of this so we can compare it to log(u) without
    ever calling exp().

    Parameters
    ----------
    log_post_proposed : float
        Log-posterior at the proposed point.
    log_post_current : float
        Log-posterior at the current point.

    Returns
    -------
    log_alpha : float
        min(0, log_post_proposed - log_post_current)
    """
    pass   # 1 line using np.minimum

In [ ]:
# --- STUDENT (3c): single MH step ---

def mh_step(theta_current, log_post_fn, step_size, **kwargs):
    """
    Perform one complete Metropolis-Hastings step.

    Parameters
    ----------
    theta_current : np.ndarray
        Current position in parameter space.
    log_post_fn : callable
        Function returning the log-posterior. Signature:
            log_post_fn(theta, **kwargs) -> float
    step_size : float or np.ndarray
        Passed to propose().
    **kwargs
        Passed through to log_post_fn (e.g. data=...).

    Returns
    -------
    theta_next : np.ndarray
        Next position: the proposal if accepted, theta_current if rejected.
    accepted : bool
        True if the proposal was accepted.
    """
    pass   # ~5 lines:
           # 1. call propose() to get theta_proposed
           # 2. evaluate log_post_fn at both points
           # 3. call acceptance_probability()
           # 4. draw u ~ Uniform(0, 1)
           # 5. compare log(u) to log_alpha; return accordingly

In [ ]:
# --- PROVIDED: master chain runner ---
# Bring your three functions together into a full chain.

def run_mh_chain(log_post_fn, theta_init, step_size, n_steps, **kwargs):
    """
    Run an MH chain for n_steps starting at theta_init.

    Parameters
    ----------
    log_post_fn : callable
        Log-posterior function, passed to mh_step.
    theta_init : array-like, shape (n_params,)
        Starting point. Must lie within the prior support.
    step_size : float or np.ndarray
        Proposal standard deviation.
    n_steps : int
        Total number of iterations (includes burn-in).
    **kwargs
        Passed through to log_post_fn via mh_step.

    Returns
    -------
    chain : np.ndarray, shape (n_steps, n_params)
        All samples including burn-in.
    acceptance_rate : float
        Fraction of proposed moves that were accepted.
    """
    theta_init = np.atleast_1d(np.asarray(theta_init, dtype=float))
    n_params   = len(theta_init)
    chain      = np.zeros((n_steps, n_params))
    chain[0]   = theta_init
    n_accepted = 0

    for i in tqdm(range(1, n_steps), desc='MCMC', unit='step'):
        chain[i], accepted = mh_step(chain[i - 1], log_post_fn,
                                     step_size, **kwargs)
        n_accepted += int(accepted)

    return chain, n_accepted / n_steps

In [ ]:
# --- PROVIDED: run your sampler on the toy posterior ---
# If the acceptance rate is near 0 or 1, check your implementation.

initial_theta = np.array([0.0, 0.0])
step_size     = 1.5
n_steps       = 10000

np.random.seed(42)
print(f'Run | initial_theta={initial_theta}, step_size={step_size}, n_steps={n_steps}')
chain, acc_rate = run_mh_chain(
    log_posterior_toy,
    theta_init = initial_theta,
    step_size  = step_size,
    n_steps    = n_steps,
)

print(f'Chain shape:     {chain.shape}')
print(f'Acceptance rate: {acc_rate:.2f}   (target: 0.25 – 0.45)')

---
## Section 4 — Diagnostics & building intuition 

A chain that runs without errors is not necessarily a *good* chain. These diagnostics tell you whether to trust your samples.

> **Complete Section 3 before running these cells.**

### Initial guesses 

A healthy chain looks like white noise after the burn-in period. 

Go back to the previous cell and start from better guesses and far poorer guesses. Remember, we know the truth to be at $\boldsymbol{\mu} = [3,\,-1]$ . How do better and worse guesses impact how quickly we reach burn-in? 

What do you therefore need to keep in mind for burn-in when exploring unknown regions of parameter space (where your initial guess might be poor) vs. exploring regions where solutions are known to lie? 

How important is it to provide a good guess to the sampler? Is this expected to scale with dimensionality / noise / non-Guassianity ?

In [ ]:
# --- PROVIDED: trace plots ---

N_BURN = 2000 
# 30% burn-in is a common choice, but you can experiment with this. 
# Try 1000 or 5000 and see how it affects the trace plots 
# and the posterior estimates in the next cell.

PARAM_LABELS = [r'$\theta_1$', r'$\theta_2$']

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(chain[:, i], lw=0.5, color='steelblue', alpha=0.8, 
            label='chain' if i == 0 else '')
    ax.axvline(N_BURN, color='red', ls='--', lw=1.5,
               label='burn-in cut' if i == 0 else '')
    ax.axhline(MU_TOY[i], color='k', ls=':', lw=1, alpha=0.5,
               label='truth' if i == 0 else '')
    ax.set_ylabel(PARAM_LABELS[i])

axes[-1].set_xlabel('MCMC step')
axes[0].legend(loc=(0.27, 1.1), ncols=3, frameon=False)
fig.suptitle(f'Trace plots  |  acceptance rate = {acc_rate:.2f}', fontsize=12)
plt.tight_layout()
plt.show()
print(f'Post-burn-in samples: {len(chain) - N_BURN}')

### Step size and acceptance rates 


Let's next explore the effect of step sizes. 

Here are three chains with very different step sizes. The y-axes show their acceptance rates. 

While a 95% acceptance rate might sound good, does it look like the chain is effectively exploring the parameter space? Does the next point depend on the previous point? Therefore, are the points in the chain independent or correlated? 

What about the other extreme - what are the problems with the chain with too large a step-size and too low an acceptance rate? Is this effectively exploring the parameter space? 

Which of the three chains looks like white noise (and therefore burnt-in)? 

In [ ]:
# --- PROVIDED: effect of proposal step size ---

step_sizes = [0.05,   1.0,   20.0]
labels     = ['Too narrow  (σ=0.05)', 'About right  (σ=1.0)', 'Too wide  (σ=20.0)']
colors     = ['darkorange', 'steelblue', 'firebrick']

initial_theta = np.array([0, 0])
n_steps       = 3000

print(f'Run | initial_theta={initial_theta}, step_sizes={step_sizes}, n_steps={n_steps}')

fig, axes = plt.subplots(3, 2, figsize=(12, 7))
for row, (ss, lbl, col) in enumerate(zip(step_sizes, labels, colors)):
    np.random.seed(0)
    c, ar = run_mh_chain(log_posterior_toy, np.array(initial_theta), ss, n_steps)
    for j in range(2):
        axes[row, j].plot(c[:, j], lw=0.5, color=col)
        axes[row, j].axhline(MU_TOY[j], color='k', ls=':', lw=1, alpha=0.4)
    axes[row, 0].set_ylabel(f'{lbl}\nacc={ar:.2f}', fontsize=12)
    if row == 0:
        axes[0, 0].set_title(r'$\theta_1$ trace')
        axes[0, 1].set_title(r'$\theta_2$ trace')
for ax in axes[-1]:
    ax.set_xlabel('Iteration')
plt.suptitle('Effect of proposal step size on chain behaviour', fontsize=12)
plt.tight_layout()
plt.show()

### Step size, autocorrelaiton and effective number of samples 


Let's actually plot the autocorrelation of the chain for different choices of `step_size` and `initial_theta`. 

Which of these impacts the autocorrelation length? Why? Would this change if we didn't remove burn-in? 

What does it mean if you have correlated samples? Why would independent samples be preferrable? 

The autocorrelation is expected to to roughly decay exponentially as $e^{-c/\tau}$. So to find the autocorrelation length, we check where the autocorrelation crosses $e^{-1}$, i.e. the point at which $c = \tau$ graphically. Knowing the correlation length, we can determine the number of independent samples as 

$$\frac{N_{steps} - N_{burn-in}}{\tau}$$

Summing over autocorrelations in both the 'past' and 'future' directions of every step in the chain. 



In [ ]:
# --- PROVIDED: autocorrelation and effective sample size ---

N_BURN = 2000

initial_theta = (10, 10)   # start away from the mean; we discard N_BURN below
step_size = 1.5
n_steps = 10000

print(f'Run | initial_theta={initial_theta}, step_size={step_size}, n_steps={n_steps}, N_BURN={N_BURN}')
chain, acc_rate = run_mh_chain(
    log_posterior_toy,
    theta_init = np.array(initial_theta),
    step_size  = step_size,
    n_steps    = n_steps,
)

samples = chain[N_BURN:]

def autocorr_func(x, max_lag=400):
    """Normalised autocorrelation of 1D array x."""
    x  = x - x.mean()              # centre: required for a proper correlation
    c  = np.correlate(x, x, mode='full')  # full autocorrelation: length 2N-1, lags -(N-1)..+(N-1)
    c  = c[c.size // 2:]           # keep only lags 0, 1, 2, ... (right half; left is symmetric)
    c /= c[0]                      # normalise by zero-lag (= variance) s.t. lag-0 value is 1
    return c[:max_lag]             # truncate: long-lag values are noisy and rarely informative

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, ax in enumerate(axes):
    acf  = autocorr_func(samples[:, i])
    lags = np.arange(len(acf))
    ax.plot(lags, acf, color='steelblue', lw=1.2)
    ax.axhline(0,      color='k',   lw=0.5)
    ax.axhline(1/np.e, color='red', ls='--', lw=1,
               label='1/e  (rough \u03c4 estimate)')
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(PARAM_LABELS[i])
    ax.legend()
plt.suptitle('Autocorrelation functions (post burn-in)', fontsize=12)
plt.tight_layout()
plt.show()

print('Effective sample sizes:')
for i in range(2):
    acf  = autocorr_func(samples[:, i])
    # find where the autocorrelation drops below 1/e
    drop = np.argwhere(acf < 1 / np.e) 
    # if it never drops below 1/e, use the full length of acf
    tau  = int(drop[0, 0]) if len(drop) else len(acf) 
    # effective sample size: N_eff = N_post_burnin / (2 * tau)
    N_eff = len(samples) / (2 * tau)
    print(f'  {PARAM_LABELS[i]}:  \u03c4 \u2248 {tau},  '
          f'N_eff \u2248 {N_eff:.0f}  (out of {len(samples)} samples)')
print(f'\nFor acceptance rate {acc_rate:.2f}')
print('\nN_eff < N because sequential samples are correlated.')

### Convergence 

We practically never run just one chain and rely exclusively on that to have sufficiently explored the parameter space. Running multiple chains has the advantage that they will randomly-walk differently and explore different parts of the parameter space. But if all chains are in different corners of the parameter space, where do the true underlying parameters lie? 

It is therefore useful to consider a convergence statistic that can tell us if chains have explored the same regions of parameter space even if they did it in different sequences. This can tell us that our results are robust to initial-point biases or local minima. 

The [Gelman-Rubin criterion](https://en.wikipedia.org/wiki/Gelman-Rubin_statistic) $R$ is the most-used diagnostic, which compares the variance within chains to the variance across chains. This checks whether all chains agree with each other. If multiple chains started from very different initial points, yet they all converged to the same distribution, their internal spread should match the spread between them. 

Usually, we care about $R-1$ and want this to be small $\lesssim 0.05$ for robust chains. 

Try different `start_points` and `step_size` for the chains. Which do you expect will impact convergence? Why? 

In [ ]:
# --- PROVIDED: Gelman-Rubin convergence diagnostic ---
# Runs multiple chains from different starting points.
# R-hat near 1 means all chains converged to the same distribution.

def gelman_rubin(chains):
    """
    Compute the Gelman-Rubin R-hat statistic.

    Parameters
    ----------
    chains : np.ndarray, shape (n_chains, n_steps, n_params)

    Returns
    -------
    R_hat : np.ndarray, shape (n_params,)
    """
    M, N, D   = chains.shape                  # M chains, N samples each, D parameters
    mu_chains = chains.mean(axis=1)           # mean of each chain: shape (M, D)
    mu_grand  = mu_chains.mean(axis=0)        # grand mean across all chains: shape (D,)
    B  = N / (M - 1) * np.sum((mu_chains - mu_grand) ** 2, axis=0)
        # between-chain variance: shape (D,)
    W  = np.mean(np.var(chains, axis=1, ddof=1), axis=0)
        # within-chain variance: shape (D,)
    var_hat = (N - 1) / N * W + B / N
        # weighted sum of within-chain and between-chain variance: shape (D,)
    return np.sqrt(var_hat / W)
        # noramlise by within-chain variance 


# Choose your starting points, step size for the 4 chains 

start_points = [
    np.array([-2., -5.]),
    np.array([ 8.,  4.]),
    np.array([ 0.,  3.]),
    np.array([ 6., -5.]),
]

step_size = 1.0

n_steps = 5000

N_BURN = 2000

print(f'Run | {len(start_points)} chains, step_size={step_size}, n_steps={n_steps}, N_BURN={N_BURN}')

all_chains = []
acceptance_rates = []
for k, start in enumerate(start_points):
    np.random.seed(k * 17 + 3)
    c, acc_rate = run_mh_chain(log_posterior_toy, start, step_size, n_steps)
    all_chains.append(c[N_BURN:])
    acceptance_rates.append(acc_rate)

print(f'\nAcceptance rate for each chain: {[acc_rate for acc_rate in acceptance_rates]}\n')

R_hat = gelman_rubin(np.array(all_chains))

print('Gelman-Rubin R-1  (target: < 0.05):')
for i, r in enumerate(R_hat):
    status = '\u2713  converged' if r < 1.05 else '\u2717  NOT CONVERGED'
    print(f'  {PARAM_LABELS[i]}:  R-1 = {r-1.:.4f}   {status}')

In [ ]:
# --- PROVIDED: check scatter to see convergence ---

def log_likelihood_toy(theta):
    """Log-likelihood of the 2D correlated Gaussian."""
    d = np.asarray(theta, dtype=float) - MU_TOY
    return float(_LOG_NORM - 0.5 * d @ _COV_INV @ d)

# grid to get likelihood contours
t1 = np.linspace(-1.5, 7.5, 200); t2 = np.linspace(-8.0, 6.0, 200)
T1, T2 = np.meshgrid(t1, t2)
Z = np.vectorize(lambda a, b: log_likelihood_toy([a, b]))(T1, T2)
levels = Z.max() - np.array([4.5, 2.0, 0.5])

fig, ax = plt.subplots(1, 1, figsize=(6, 5), sharex=True, sharey=True)

# left: a posterior cloud (drawn directly from the Gaussian for illustration)
ax.contour(T1, T2, Z, levels=levels, colors='0.7', linewidths=1)
for c,col,cnum in zip(all_chains, ['gold', 'greenyellow', 'turquoise', 'dodgerblue'], range(1,5)):
    ax.plot(c[:, 0], c[:, 1], '.', ms=2, color=col, alpha=0.2,
            label=f'chain {cnum}')


ax.set_xlabel(PARAM_LABELS[0])
ax.set_ylabel(PARAM_LABELS[1])
# ax.legend(loc='upper left', frameon=False, fontsize=10)
plt.tight_layout(); plt.show()

---
## Section 5 — Validate your sampler against the known answer

Let's up together everything you've learnt so far. 

If your sampler is correct, the sample statistics should match the analytical 2D Gaussian.

If they don’t match: 
1. did you discard burn-in? 
2. do you have enough steps? 
3. is acceptance rate in range?

In [ ]:
# --- PROVIDED: corner plot vs analytical posterior ---

initial_theta = (10,10)  
step_size = 1.5
n_steps = 10000
N_BURN = 2000

print(f'Run | initial_theta={initial_theta}, step_size={step_size}, n_steps={n_steps}, N_BURN={N_BURN}')
chain, acc_rate = run_mh_chain(
    log_posterior_toy,
    theta_init = np.array(initial_theta),
    step_size  = step_size,
    n_steps    = n_steps,
)

samples = chain[N_BURN:]

print(f'\nAcceptance rate: {acc_rate:.2f}   (target: 0.25 – 0.45)')
print(f'Post-burn-in samples: {len(samples)} \nRemoved {N_BURN/n_steps*100:.1f}% for burn-in')

fig = corner.corner(
    samples,
    labels       = PARAM_LABELS,
    truths       = MU_TOY,
    truth_color  = 'red',
    color        = 'steelblue',
    bins         = 30,
    show_titles  = True,
    title_fmt    = '.2f',
    title_kwargs = {'fontsize': 10},
)
fig.suptitle('MCMC samples vs analytical truth  (red = true mean)', y=1.01, fontsize=12)
plt.show()

print('Sample mean:     ', samples.mean(axis=0).round(3))
print('Analytical mean: ', MU_TOY)
print()
print('Sample covariance:')
print(np.cov(samples.T).round(3))
print('Analytical covariance:')
print(COV_TOY)

**Discussion**

- What happens if you reduce `n_steps` to 500?
- What happens if you forget to discard burn-in (`N_BURN = 0`)?
- What happens if you run with the too-narrow step size?

These are the mistakes that invalidate real cosmological analyses.

---
## Section 6 — Cosmological application: mock Type Ia supernovae

You will now apply the *same* `run_mh_chain` function — unchanged — to a cosmological problem.

**The model.** At low redshift, the distance modulus of a Type Ia supernova is:

$$\mu_{\rm th}(z \mid H_0, M) = 5\log_{10}\!\left(\frac{c\,z}{H_0}\right) + 25 + M$$

where $H_0$ is the Hubble constant and $M$ is the SN absolute magnitude. Each observed $\mu_i$ has Gaussian scatter $\sigma_i$.

The likelihood and prior are **provided** (from the data-analysis class). Your jobs:
1. Set the true parameters and generate mock data.
2. Combine the provided likelihood and prior into a posterior (2 lines).
3. Run your sampler and check that you recover the truth you set.

In [ ]:
# ============================================================
# SET YOUR TRUE PARAMETERS HERE
# Your sampler should recover these values.
# Try changing them and re-running everything below.
# ============================================================
H0_true   = 70.0    # Hubble constant      [km/s/Mpc]
M_true    = -19.3   # SN Ia abs. magnitude [mag]
N_SNE     = 25      # number of mock supernovae
SIGMA_OBS = 0.15    # photometric scatter   [mag]
Z_MIN     = 0.02    # redshift range (low-z: Hubble law valid)
Z_MAX     = 0.15
# ============================================================

C_LIGHT = 2.998e5   # speed of light [km/s]
np.random.seed(42)

z_obs  = np.sort(np.random.uniform(Z_MIN, Z_MAX, N_SNE))
mu_th  = 5 * np.log10(C_LIGHT * z_obs / H0_true) + 25 + M_true
mu_obs = mu_th + np.random.normal(0, SIGMA_OBS, N_SNE)
mu_err = np.full(N_SNE, SIGMA_OBS)

data = dict(z=z_obs, mu_obs=mu_obs, mu_err=mu_err,
            H0_true=H0_true, M_true=M_true)

print(f'Generated {N_SNE} mock SNe Ia')
print(f'True parameters:  H0 = {H0_true} km/s/Mpc,  M = {M_true}')

In [ ]:
# --- PROVIDED: plot the mock data ---

z_plt  = np.linspace(Z_MIN, Z_MAX, 300)
mu_plt = 5 * np.log10(C_LIGHT * z_plt / H0_true) + 25 + M_true

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(z_obs, mu_obs, yerr=mu_err,
            fmt='o', ms=5, color='steelblue', ecolor='steelblue',
            alpha=0.8, label='Mock SNe Ia')
ax.plot(z_plt, mu_plt, 'r-', lw=2,
        label=f'True model  (H\u2080={H0_true}, M={M_true})')
ax.set_xlabel('Redshift  z')
ax.set_ylabel('Distance modulus  \u03bc')
ax.set_title('Mock Type Ia supernova data')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- PROVIDED: likelihood and prior for the SN Ia problem ---
# Likelihood construction is covered in the separate data-analysis class.

def log_likelihood_sne(theta, data):
    """
    Gaussian log-likelihood for SN Ia distance moduli.

    Parameters
    ----------
    theta : array-like
        [H0] for 1-parameter run, or [H0, M] for 2-parameter run.
    data : dict
        Must contain keys 'z', 'mu_obs', 'mu_err'.

    Returns
    -------
    float
    """
    H0 = float(theta[0])
    M  = float(theta[1]) if len(theta) > 1 else data.get('M_true', -19.3)
    if H0 <= 0:
        return -np.inf
    mu_model  = 5 * np.log10(C_LIGHT * data['z'] / H0) + 25 + M
    residuals = (data['mu_obs'] - mu_model) / data['mu_err']
    return -0.5 * float(np.sum(residuals ** 2))


def log_prior_cosmo(theta):
    """
    Flat prior over physically motivated parameter ranges.

    Parameters
    ----------
    theta : array-like
        [H0] or [H0, M]
    """
    H0 = float(theta[0])
    M  = float(theta[1]) if len(theta) > 1 else -19.3
    if 20.0 < H0 < 150.0 and -22.0 < M < -17.0:
        return 0.0
    return -np.inf

In [ ]:
# --- STUDENT: combine prior and likelihood into the posterior ---

def log_posterior_cosmo(theta, data):
    """
    Log-posterior for the SN Ia cosmological problem.

    Parameters
    ----------
    theta : np.ndarray
        [H0] for a 1D run, or [H0, M] for a 2D run.
    data : dict
        As returned by the mock-data cell above.

    Returns
    -------
    float
        log_prior_cosmo(theta) + log_likelihood_sne(theta, data).
        Return -np.inf immediately if the prior is -np.inf.
    """
    pass   # ~2-3 lines

In [ ]:
# --- STUDENT: run your sampler on the cosmological problem ---
# Start away from the truth. Tune step_size_cosmo until acceptance rate is ~0.25-0.45.

theta_init_cosmo = np.array([65.0])   # starting guess for H0 [km/s/Mpc]
step_size_cosmo  = 3.5                # tune this
N_STEPS_COSMO    = 15000
N_BURN_COSMO     = 3000

np.random.seed(7)
chain_cosmo, acc_cosmo = run_mh_chain(
    log_posterior_cosmo,
    theta_init_cosmo,
    step_size_cosmo,
    N_STEPS_COSMO,
    data = data,
)

print(f'Acceptance rate: {acc_cosmo:.2f}')
if 0.25 <= acc_cosmo <= 0.45:
    print('✓  In target range (0.25 – 0.45)')
else:
    print('←  Adjust step_size_cosmo and re-run')

In [ ]:
# --- PROVIDED: posterior on H0 with truth marked ---

samples_H0 = chain_cosmo[N_BURN_COSMO:, 0]
median_H0  = np.median(samples_H0)
lo68, hi68 = np.percentile(samples_H0, [16, 84])
lo95, hi95 = np.percentile(samples_H0, [ 2.5, 97.5])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples_H0, bins=60, density=True,
        color='steelblue', alpha=0.7, label='Posterior')
ax.axvspan(lo95, hi95, alpha=0.10, color='steelblue', label='95% CI')
ax.axvspan(lo68, hi68, alpha=0.20, color='steelblue', label='68% CI')
ax.axvline(data['H0_true'], color='red', lw=2.0, ls='--',
           label=f"Truth  H\u2080 = {data['H0_true']:.1f}")
ax.axvline(median_H0, color='k', lw=1.5,
           label=f'Posterior median = {median_H0:.1f}')
ax.set_xlabel('H\u2080  [km/s/Mpc]')
ax.set_ylabel('Posterior density')
ax.set_title('Recovered H\u2080 posterior from mock SN Ia data')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Truth:     H0 = {data['H0_true']:.1f} km/s/Mpc")
print(f"Posterior: {median_H0:.1f}  [{lo68:.1f}, {hi68:.1f}]  (median, 68% CI)")
recovered = lo68 <= data['H0_true'] <= hi68
print(f"Truth inside 68% CI: {'YES ' if recovered else 'NO  -- check sampler or increase N_STEPS_COSMO'}")
print()
print('Try changing H0_true in the mock-data cell and re-running. Does the posterior follow?')

In [ ]:
# # Corner plot for (H0, M) from the cosmology chain
# samples_cosmo = chain_cosmo[N_BURN_COSMO:, :2]

# fig = corner.corner(
#     samples_cosmo,
#     labels=[r"$H_0$", r"$M$"],
#     truths=[data["H0_true"], data["M_true"]],
#     truth_color="red",
#     color="steelblue",
#     bins=40,
#     show_titles=True,
#     title_fmt=".2f",
# )

# fig.suptitle("Posterior corner plot: $H_0$ and $M$", y=1.02)
# plt.show()

In [ ]:
# # Covariance matrix for (H0, M) using post-burn-in samples
# samples_H0_M = chain_cosmo[N_BURN_COSMO:, :2]
# cov_H0_M = np.cov(samples_H0_M, rowvar=False)

# print("Covariance matrix for [H0, M]:")
# print(cov_H0_M)

# # Optional: labeled output
# print(f"\nVar(H0)   = {cov_H0_M[0,0]:.4f}")
# print(f"Cov(H0,M) = {cov_H0_M[0,1]:.4f}")
# print(f"Var(M)    = {cov_H0_M[1,1]:.4f}")

## 7. A real cosmological likelihood: the LCDM two-fluid CMB spectrum

So far your sampler has explored analytic toy posteriors and a one/two-parameter
supernova model. Now we point *exactly the same* `run_mh_chain` at a real
cosmological likelihood: the **two-fluid CMB temperature (TT) power spectrum**
solver you met in the previous (CMB) class. Nothing about the
Metropolis-Hastings machinery changes -- `run_mh_chain`, `mh_step`, and
`acceptance_probability` are untouched.

What *does* change is the parameter space. We now sample the five LCDM parameters

$$\theta = \big[\ \log_{10}(10^9 A_s),\ \omega_{\rm cdm},\ \omega_b,\ H_0,\ n_s\ \big],$$

and these have two properties that break the single `step_size` knob:

1. **Wildly different scales** -- the posterior width of $\omega_b$ is
   $\sim\!10^{-4}$ while that of $H_0$ is $\sim\!1$, four orders of magnitude
   apart.
2. **Strong correlations** -- the parameters are far from independent; some pairs
   are degenerate at the $|\rho|\!\sim\!0.95$ level.

The fix is to promote the scalar `step_size` into a **`jumping_factor` $\times$
covariance matrix** proposal. Because `run_mh_chain` passes `step_size` straight
through to `propose`, the *only* function we touch is `propose` -- the master
runner never knows the difference.

> **Convergence note.** A single LCDM likelihood call takes ~0.1 s, so running
> several chains to convergence for a Gelman-Rubin $R$ test is expensive.
> We keep the multi-chain $R$ demonstration on the cheap models above and
> treat this section as a single-chain study focused on the proposal covariance.
> Expect the sampling cells below to take a few minutes in total.

### 7.1 Connecting to the solver

The TT spectrum comes from the other instructor's teaching package,
[`two_fluid_LCDM_colab`](https://github.com/tsmith2/GGI_cosmology_notebooks). The
setup cell below clones that repo and builds its small C++ solver (a few seconds,
no external libraries), then imports the ready-made likelihood. On **Google
Colab** this needs no configuration, and the solver process is created and kept
alive for you. The first likelihood call builds a small Bessel cache (~5 s); later
calls reuse it.

In [ ]:
# --- PROVIDED: get the two-fluid LCDM solver and import its likelihood ---
# On Colab this clones the other instructor's repo and builds the small C++
# solver; locally it clones next to your working directory. The solver process is
# started and kept alive by the wrapper -- if a long run is interrupted, the next
# call restarts it automatically, so there is nothing for you to manage. Point
# REPO_DIR at an existing clone to skip the download.
import sys, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/tsmith2/GGI_cosmology_notebooks.git'
IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/GGI_cosmology_notebooks') if IN_COLAB else (Path.cwd() / 'GGI_cosmology_notebooks')
PACKAGE_DIR = REPO_DIR / 'two_fluid_LCDM_colab'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['make', '-C', str(PACKAGE_DIR / 'cpp')], check=True)

sys.path.insert(0, str(PACKAGE_DIR / 'python'))
from two_fluid_LCDM_colab import (
    load_mock_data, log_prior_lcdm, log_likelihood_lcdm, model_bandpowers,
    tt_spectrum, plot_power_spectrum, FID_THETA, LCDM_BOUNDS, PARAM_NAMES,
)
print('two-fluid LCDM solver ready (H0 parameterisation)')

In [ ]:
# --- PROVIDED: mock data and parameter metadata (H0 parameterisation) ---
import numpy as np

data = load_mock_data()                          # dict with 'd_obs', 'd_sigma', 'center'
LCDM_NAMES  = ['log10(1e9 As)', 'omega_cdm', 'omega_b', 'H0', 'n_s']
LCDM_LABELS = PARAM_NAMES                         # LaTeX labels from the package ($H_0$, ...)
LCDM_FID    = FID_THETA.copy()                    # [log10(1e9 As), omega_cdm, omega_b, H0, n_s]
LCDM_NDIM   = len(LCDM_FID)
# LCDM_BOUNDS (imported): the H0 box is [55, 80] km/s/Mpc

print(f'{len(data["d_obs"])} TT bandpowers; H0 fiducial = {LCDM_FID[3]:.1f} km/s/Mpc')

### 7.2 The likelihood, prior, and posterior

For each parameter vector the solver returns a TT spectrum, which we bin into the
same bandpowers as the data and compare with a Gaussian $\chi^2$:

$$\ln\mathcal{L}(\theta) = -\tfrac12 \sum_i
  \left(\frac{D_i^{\rm obs} - D_i^{\rm model}(\theta)}{\sigma_i}\right)^2 .$$

The prior is a flat box over physically sensible ranges -- the same role as the
`log_prior_cosmo` box in the supernova example. As before, the *construction* of
the likelihood belongs to the data-analysis class; here it is provided so you can
focus on the sampler.

In [ ]:
# --- PROVIDED: prior, model, and likelihood come from the package (imported above) ---
#   log_prior_lcdm(theta)            -- flat top-hat prior over LCDM_BOUNDS (H0 box [55, 80])
#   model_bandpowers(theta)          -- C++ two-fluid solver -> binned TT bandpowers
#   log_likelihood_lcdm(theta, data) -- Gaussian bandpower chi-square vs the mock data
# Same signatures as the hand-written versions before, so nothing downstream changes.
print('provided by the package:',
      ', '.join(f.__name__ for f in (log_prior_lcdm, model_bandpowers, log_likelihood_lcdm)))

In [ ]:
# --- STUDENT: combine the prior and likelihood into the LCDM log-posterior ---

def log_posterior_lcdm(theta, data):
    """
    Log-posterior for the LCDM TT fit.

    Parameters
    ----------
    theta : np.ndarray, shape (5,)
        [log10(1e9 As), omega_cdm, omega_b, h, n_s]
    data : dict
        Provides 'd_obs', 'd_sigma' (and 'centers').

    Returns
    -------
    float
        log_prior_lcdm(theta) + log_likelihood_lcdm(theta, data),
        short-circuiting to -np.inf when the point is outside the prior box (so
        you never pay for a solver call you do not need).
    """
    pass   # ~3 lines: evaluate the prior; if it is -inf, return -inf;
           #           otherwise return prior + likelihood
           # (mirrors your log_posterior_cosmo from the supernova example)

In [ ]:
# --- PROVIDED: sanity check -- the fiducial model should fit at chi2/dof ~ 1 ---
import time

log_posterior_lcdm(LCDM_FID, data)               # first call warms the Bessel cache
_t = time.perf_counter()
_chi2 = -2.0 * log_posterior_lcdm(LCDM_FID, data)
_dt = time.perf_counter() - _t
_ndof = len(data['d_obs'])
print(f'chi2(fiducial) = {_chi2:.1f}   dof = {_ndof}   chi2/dof = {_chi2 / _ndof:.2f}')
print(f'one likelihood call: {1000 * _dt:.0f} ms')

### 7.3 Why a single `step_size` cannot work

Look at the fiducial values and their approximate posterior widths:

| parameter | fiducial | posterior width |
|---|---:|---:|
| $\log_{10}(10^9 A_s)$ | 0.322 | ~0.002 |
| $\omega_{\rm cdm}$ | 0.120 | ~0.001 |
| $\omega_b$ | 0.0223 | ~0.0001 |
| $H_0$ | 67 | ~0.6 |
| $n_s$ | 0.965 | ~0.003 |

A single scalar `step_size` must serve all five at once. Too large and every
*joint* proposal overshoots the tight, correlated region and is rejected; too
small and the wide parameters ($H_0$, $\log A_s$) only crawl. Watch it fail.

In [ ]:
# --- PROVIDED: no single scalar step_size works ---
# Short chains from the fiducial point; watch acceptance and how far H0 moves.
np.random.seed(0)
for s in [0.006, 0.001, 0.0001]:
    _chain, _acc = run_mh_chain(log_posterior_lcdm, LCDM_FID, s, 200, data=data)
    _distinct = len(np.unique(_chain[:, 3]))          # distinct values of H0 visited
    print(f'step_size={s:<8}: acceptance={_acc:.2f}   distinct H0 visited={_distinct:3d}/200  \
           max H0 difference = {np.max(_chain[:, 3]) - np.min(_chain[:, 3]):.4f}')

### 7.4 From a step size to a proposal covariance

The cure has two parts:

- **Different widths per parameter** -- propose $\Delta\theta_j$ with its own
  scale, not one shared number. That is a *diagonal* covariance.
- **Follow the correlations** -- the parameters are degenerate, so the efficient
  proposal moves *along* the degeneracy directions. That needs the *off-diagonal*
  terms too, i.e. the full covariance matrix $C$.

The Metropolis proposal becomes a multivariate Gaussian,
$\Delta\theta \sim \mathcal{N}(0,\, f^2 C)$, where the **jumping factor**
$f = 2.38/\sqrt{d}$ (here $d=5$) is the standard optimal-scaling choice for a
Gaussian target.

Because `run_mh_chain` hands `step_size` straight to `propose`, we implement all
of this by teaching `propose` to accept a 2-D array (a covariance) in the same
slot. A scalar or 1-D array keeps its old meaning, so every earlier example still
runs unchanged.

In [ ]:
# --- STUDENT: generalise `propose` to accept a covariance matrix ---
# This REPLACES your earlier propose. It must stay backward compatible:
#   * scalar or 1-D step_size  -> the old per-parameter Gaussian (np.random.normal)
#   * 2-D step_size (a matrix) -> a full multivariate Gaussian proposal
# run_mh_chain and mh_step do NOT change.

def propose(theta_current, step_size):
    """
    Draw a candidate from a Gaussian proposal centred on theta_current.

    step_size : float or np.ndarray
        - scalar / 1-D : standard deviation(s); independent per parameter.
        - 2-D (n x n)  : the proposal COVARIANCE matrix (correlations included).

    Returns
    -------
    theta_proposed : np.ndarray
    """
    pass   # branch on np.ndim(step_size):
           #   == 2 -> np.random.multivariate_normal(mean=..., cov=...)
           #   else -> np.random.normal(loc=..., scale=...)   (your original line)

In [ ]:
# --- PROVIDED: a short pilot chain with a DIAGONAL proposal ---
# Rough per-parameter widths -- an order-of-magnitude first guess, no correlations.
diag_widths = np.array([0.004, 0.0015, 0.0002, 0.6, 0.004])

# Deliberately small steps (0.3x) so the strongly-correlated chain still moves and
# traces out the posterior shape we will estimate the covariance from.
pilot_cov = np.diag((0.3 * diag_widths) ** 2)

np.random.seed(1)
N_PILOT = 800
pilot_chain, pilot_acc = run_mh_chain(log_posterior_lcdm, LCDM_FID,
                                      pilot_cov, N_PILOT, data=data)
print(f'pilot acceptance = {pilot_acc:.2f}   (it moves, but -- as we will see -- mixes slowly)')

In [ ]:
# Corner plot for LCDM parameters from the pilot chain
samples_cosmo = pilot_chain[200:, :]

fig = corner.corner(
    samples_cosmo,
    labels=LCDM_LABELS,
    truths=LCDM_FID,
    truth_color="red",
    color="steelblue",
    bins=40,
    show_titles=True,
    title_fmt=".2f",
)

fig.suptitle("Posterior corner plot: LCDM pilot chain", y=1.02)
plt.show()

### 7.5 Estimate the covariance and run the tuned chain

The pilot chain has already sketched the shape of the posterior. Estimate its
covariance $C$, scale by $f^2$, and hand that matrix to `run_mh_chain` in the
`step_size` slot. Inspecting the correlation matrix shows exactly what a diagonal
proposal was throwing away.

In [ ]:
# --- STUDENT: build the proposal covariance from the pilot chain ---
burn_pilot = 200
samples = pilot_chain[burn_pilot:]

# 1. Estimate the 5x5 parameter covariance C from `samples`.
#    np.cov treats each ROW as a variable, so mind the orientation of `samples`
#    (it is N_steps x 5).
C = None

# 2. Set the optimal-scaling jumping factor for a d-dimensional Gaussian target:
#    f = 2.38 / sqrt(d), with d = LCDM_NDIM.
jumping_factor = None

# 3. Form the proposal covariance you will pass to run_mh_chain:  f**2 * C.
proposal_cov = None

# Once C is set, uncomment to see the correlations a diagonal proposal ignores:
# _D = np.sqrt(np.diag(C)); print(np.round(C / np.outer(_D, _D), 2))

In [ ]:
# --- STUDENT: run the covariance-tuned chain ---
# Start from the end of the pilot (already near the posterior bulk).
np.random.seed(2)
N_COV = 1500                      # bump up (e.g. 8000) for a production-quality corner

chain_lcdm, acc_lcdm = None, None      # call run_mh_chain with proposal_cov in the
                                       # step_size slot, theta_init = pilot_chain[-1],
                                       # n_steps = N_COV, and data=data

print(f'covariance-tuned acceptance = {acc_lcdm:.2f}   (aim ~0.2 - 0.4)')

In [ ]:
# --- PROVIDED: mixing -- diagonal pilot vs covariance-tuned chain ---
import matplotlib.pyplot as plt

burn = 200
fig, axes = plt.subplots(LCDM_NDIM, 1, figsize=(9, 8), sharex=True)
for j, ax in enumerate(axes):
    ax.plot(chain_lcdm[:, j], lw=0.7, color='steelblue')
    ax.axhline(LCDM_FID[j], color='k', ls=':', lw=1)
    ax.set_ylabel(LCDM_LABELS[j])
axes[0].set_title('covariance-tuned chain -- traces (dotted = input truth)')
axes[-1].set_xlabel('step')
plt.tight_layout(); plt.show()

def _tau(x):
    """Autocorrelation length: first lag where the ACF drops below 1/e (as in Sec. 5)."""
    acf = autocorr_func(x)
    drop = np.argwhere(acf < 1 / np.e)
    return int(drop[0, 0]) if len(drop) else len(acf)

print('autocorrelation length  (lower = more independent samples):')
for j in range(LCDM_NDIM):
    print(f'  {LCDM_NAMES[j]:<13}: diagonal ~{_tau(pilot_chain[burn_pilot:, j]):4d}'
          f'    covariance ~{_tau(chain_lcdm[burn:, j]):4d}')

In [ ]:
# --- PROVIDED: posterior corner plot vs the input truth ---
import corner

fig = corner.corner(chain_lcdm[burn:], labels=LCDM_LABELS, truths=LCDM_FID,
                    truth_color="red", color="steelblue",
                    show_titles=True, title_fmt='.4f')
fig.suptitle(f'LCDM TT -- covariance-tuned chain (acceptance {acc_lcdm:.2f})', y=1.02)
plt.show()

### 7.6 What the covariance proposal bought you

- A single scalar `step_size` was hopeless: too large froze the chain, too small
  let it only crawl.
- A **diagonal** proposal fixed the *scales* but still fought the parameter
  **correlations**, so it mixed slowly -- a long autocorrelation and few
  effective samples.
- A **`jumping_factor` $\times$ covariance** proposal moves along the degeneracy
  directions. Same `run_mh_chain`, one upgraded `propose`, and an order of
  magnitude more independent samples per unit time.

This is how production samplers (CosmoMC, MontePython, Cobaya) work: they start
from an estimated parameter covariance (from a Fisher forecast or a short run)
and often keep *adapting* it as the chain explores. Estimating $C$ from a pilot
chain, as you did here, is the simplest version of that idea.

**Runtime knobs.** The chain above is deliberately short so it finishes quickly.
For a smooth, production-quality corner, raise `N_COV` (e.g. 8000); for more
model accuracy raise `N_SOURCE` (with a Bessel cache built for that grid). You can
also load a longer precomputed chain just for plotting.

**Next.** Notebook 2 keeps this same likelihood but asks a different question:
instead of *sampling* the posterior, we *profile* it -- fixing one parameter,
optimising the rest, and mapping out the profile likelihood.

**In your own time**, try going back to Section 6 and running with both $H_0$ and 
$M$. These two parameters are fully degenerate, so with just a single step size or 
even a diagonal covariance, you may not recover the inputs within 68%. But try 
calculating a covariance for the parameters from an initial chain, feed that back 
into the sampler and see if you do any better. Remember, this requires our newly 
updated `propose` function. 